<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail_Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [2]:
TRADE_CAPITAL = 5000 # Global constant for the capital allocated per trade
LOOKBACK_DAYS = 100

### Remember to load the OptionVolume.csv file before running the scripts

In [3]:
# =======================
#   SIGNAL PROFITABILITY
# =======================

def calculate_signal_profitability(df, capital_per_trade=TRADE_CAPITAL):
    """
    Calculates dollar-value profitability for unique signals
    with a 1-5 day forward lookback, applying a 5-day trading lockout.
    """
    # 1. De-duplicate signals: 1 position per stock per day max
    trade_df = df.drop_duplicates(subset=['Date', 'Symbol']).copy()
    trade_df['Date'] = pd.to_datetime(trade_df['Date']).dt.normalize() # Normalize to remove time component
    trade_df = trade_df.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)

    # 2. Fetch historical data for all relevant symbols once
    symbols_to_fetch = trade_df['Symbol'].unique()
    # Fetch enough data to cover the forward lookback from the latest signal
    # and also a few days before the earliest signal for the trading calendar
    start_fetch_date = trade_df['Date'].min() - timedelta(days=10)
    end_fetch_date = trade_df['Date'].max() + timedelta(days=10)

    print(f" ʗ Analyzing profitability for {len(trade_df)} unique signals. Fetching historical data for {len(symbols_to_fetch)} symbols...")

    historical_data = yf.download(list(symbols_to_fetch), start=start_fetch_date.strftime('%Y-%m-%d'),
                                  end=end_fetch_date.strftime('%Y-%m-%d'), progress=False, auto_adjust=True)

    if historical_data.empty:
        print("⚠️ No historical data downloaded for analysis. Returning empty report.")
        return pd.DataFrame()

    close_prices = pd.DataFrame()
    if isinstance(historical_data.columns, pd.MultiIndex):
        close_prices = historical_data['Close']
    elif len(symbols_to_fetch) == 1 and 'Close' in historical_data.columns:
        # Handle single symbol case where yf.download returns a DataFrame with 'Close' column
        close_prices = historical_data[['Close']].rename(columns={'Close': symbols_to_fetch[0]})
    elif len(symbols_to_fetch) == 1 and isinstance(historical_data, pd.Series): # Sometimes yf.download returns a Series for single symbol
        close_prices = historical_data.to_frame(name=symbols_to_fetch[0])
    else:
        print("⚠️ Could not extract 'Close' prices from historical data. Returning empty report.")
        return pd.DataFrame()

    # Ensure close_prices has a DatetimeIndex
    if not isinstance(close_prices.index, pd.DatetimeIndex):
        close_prices.index = pd.to_datetime(close_prices.index)
    close_prices.index = close_prices.index.normalize() # Normalize index
    close_prices = close_prices.sort_index()

    # 3. Implement 5-day trading lockout
    trade_df['is_valid_signal'] = True

    # Use a list to collect valid signal rows for concatenation
    valid_signal_rows = []

    for symbol in symbols_to_fetch:
        symbol_signals = trade_df[trade_df['Symbol'] == symbol].copy()

        if symbol_signals.empty:
            continue

        if symbol not in close_prices.columns:
            print(f"❌ Trading calendar not available for {symbol} to apply lockout. Skipping lockout for this symbol. All its signals will be considered valid.")
            valid_signal_rows.extend(symbol_signals.to_dict('records')) # Add all signals if lockout can't be applied
            continue

        ticker_trading_days = close_prices[symbol].dropna().index.normalize()
        ticker_trading_days = ticker_trading_days.sort_values().drop_duplicates()

        last_valid_signal_calendar_idx = -np.inf # Index in ticker_trading_days

        for i, row in symbol_signals.iterrows():
            current_signal_date = row['Date']

            # Find the position of the current signal date in the symbol's trading calendar
            actual_signal_trading_day = current_signal_date
            if current_signal_date not in ticker_trading_days:
                # If signal falls on a non-trading day, find the next trading day
                next_trading_day_locs = ticker_trading_days[ticker_trading_days > current_signal_date]
                if not next_trading_day_locs.empty:
                    actual_signal_trading_day = next_trading_day_locs.min()
                else:
                    # No future trading days, cannot determine lockout period. Mark as invalid.
                    trade_df.loc[i, 'is_valid_signal'] = False
                    continue # Skip to next signal

            current_calendar_idx = ticker_trading_days.get_loc(actual_signal_trading_day)

            # 5 trading days lockout: means the next signal must be at least 5 trading days *after* the last valid signal date
            # Example: Signal on Day 0. Lockout until Day 5. Next signal must be on Day 5 or later.
            # Calendar indices: 0 (signal), 1, 2, 3, 4 (locked out), 5 (first eligible day)
            # So, `current_calendar_idx` must be >= `last_valid_signal_calendar_idx + 5`
            if current_calendar_idx >= last_valid_signal_calendar_idx + 5:
                # This signal is valid
                trade_df.loc[i, 'is_valid_signal'] = True
                last_valid_signal_calendar_idx = current_calendar_idx
                valid_signal_rows.append(row.to_dict()) # Collect only valid rows
            else:
                # This signal is within the lockout period
                trade_df.loc[i, 'is_valid_signal'] = False

    if not valid_signal_rows:
        print("⚠️ No valid signals after applying lockout. Returning empty report.")
        return pd.DataFrame()

    # Reconstruct the DataFrame from valid signal rows
    trade_df_final = pd.DataFrame(valid_signal_rows)

    # 4. Trade Count Validation
    total_valid_signals = len(trade_df_final)
    if total_valid_signals < 30:
        print(f"⚠️ LOCKOUT ALERT: Valid Trade_Count ({total_valid_signals}) < 30 threshold after 5-day lockout. Results should be treated as anecdotal.")

    # 5. Profit calculation for valid signals
    def get_forward_profit(row):
        symbol = row['Symbol']
        sig_date = row['Date']
        # Ensure Price is a number, handling '$' if present
        entry_price = float(str(row['Price']).replace('$', '')) if isinstance(row['Price'], (str, np.str_)) else row['Price']

        # Get price series for this ticker starting from the signal date
        if symbol not in close_prices.columns:
            return pd.Series([np.nan] * 5, index=[f'Day_{i}_$' for i in range(1, 6)])

        # Ensure sig_date is in the index of ticker_series, or find the next available
        ticker_series = close_prices[symbol].loc[sig_date:].head(6) # Day 0 to Day 5 (6 prices total)

        # Calculate shares to purchase without exceeding capital_per_trade
        shares = np.floor(capital_per_trade / entry_price) if entry_price != 0 else 0

        profits = {}

        for i in range(1, 6): # Day 1 through Day 5
            if len(ticker_series) > i: # Check if the i-th trading day exists (i.e., we have enough forward data)
                exit_price = ticker_series.iloc[i]
                profits[f'Day_{i}_$'] = round((exit_price - entry_price) * shares, 2)
            else:
                profits[f'Day_{i}_$'] = np.nan # Not enough data for this forward day
        return pd.Series(profits)

    profit_results = trade_df_final.apply(get_forward_profit, axis=1)
    final_report = pd.concat([trade_df_final, profit_results], axis=1)

    # Sort the final report by Date
    final_report = final_report.sort_values(by='Date').reset_index(drop=True)

    return final_report

In [4]:
# ========================
# SUMMARIZE TRADES
# ========================

def calculate_exposure_metrics(signals_df, _unused_price_df=None, trade_size=TRADE_CAPITAL):
    """
    Calculates Max and Avg Value Risked for 1D, 2D, 3D, 4D, and 5D windows,
    representing the sum of capital for all open positions on a given day.
    """
    exposure_results = {}
    windows = [1, 2, 3, 4, 5] # User requested windows

    # Ensure 'Date' column is datetime and set as index for time series operations
    signals_for_exposure = signals_df.copy()
    signals_for_exposure['Date'] = pd.to_datetime(signals_for_exposure['Date'])
    signals_for_exposure = signals_for_exposure.set_index('Date').sort_index()

    if signals_for_exposure.empty:
        for days in windows:
            exposure_results[f'Max_Risked_{days}D'] = 0
            exposure_results[f'Avg_Risked_{days}D'] = 0
        return exposure_results

    # Ensure 'Price' column is numeric
    signals_for_exposure['Price'] = signals_for_exposure['Price'].astype(str).str.replace('$', '').astype(float)

    # Determine the full date range to ensure a continuous time series for exposure
    min_sig_date = signals_for_exposure.index.min()
    max_sig_date = signals_for_exposure.index.max()
    # The exposure can extend past the last signal date by the maximum window
    full_time_index = pd.date_range(start=min_sig_date, end=max_sig_date + timedelta(days=max(windows) - 1), freq='D')

    # Calculate actual capital invested per signal based on the 'Price' column
    # The 'Price' column in signals_df (profit_df) is already a float after calculate_signal_profitability
    shares_per_signal = np.floor(trade_size / signals_for_exposure['Price'])
    actual_capital_per_signal = shares_per_signal * signals_for_exposure['Price']

    # Create a Series where index is date and value is actual capital added on that date
    daily_capital_inflow = actual_capital_per_signal.groupby(signals_for_exposure.index).sum()

    # Reindex daily_capital_inflow to the full_time_index, filling NaNs with 0 for days without new signals
    daily_capital_inflow = daily_capital_inflow.reindex(full_time_index, fill_value=0)

    for days in windows:
        # Initialize capital_flow_net with a float dtype to prevent FutureWarning
        capital_flow_net = pd.Series(0.0, index=full_time_index, dtype=float)
        # Capital entering on signal date
        capital_flow_net.loc[daily_capital_inflow.index] += daily_capital_inflow.values

        # Capital exiting after 'days' days. Create a temporary DataFrame to track these exits.
        # Only consider dates in daily_capital_inflow that are within the full_time_index
        valid_inflow_dates = daily_capital_inflow.index.intersection(full_time_index)
        temp_exits_df = pd.DataFrame(index=valid_inflow_dates)
        temp_exits_df['capital_to_exit'] = daily_capital_inflow.loc[valid_inflow_dates]
        temp_exits_df['exit_date'] = temp_exits_df.index + pd.Timedelta(days=days)

        # Aggregate outflows by date, as multiple trades might exit on the same day
        daily_capital_outflow = temp_exits_df.groupby('exit_date')['capital_to_exit'].sum()

        # Apply outflows to capital_flow_net, ensuring index alignment with full_time_index
        common_exit_dates = daily_capital_outflow.index.intersection(capital_flow_net.index)
        capital_flow_net.loc[common_exit_dates] -= daily_capital_outflow.loc[common_exit_dates]

        # The cumulative sum of capital_flow_net gives the total capital exposed on each day
        current_exposure_series = capital_flow_net.cumsum()

        # Ensure exposure doesn't go negative due to calculation quirks and clip to 0.
        current_exposure_series = current_exposure_series.clip(lower=0)

        # Calculate Max and Avg from the resulting daily exposure series
        exposure_results[f'Max_Risked_{days}D'] = current_exposure_series.max() if not current_exposure_series.empty else 0
        exposure_results[f'Avg_Risked_{days}D'] = current_exposure_series.mean() if not current_exposure_series.empty else 0

    return exposure_results


def display_strategy_summary(perf_df, algorithm_name, trade_size, lookback_days):
    """
    Architectural Grade Summary: Restores Alpha Drivers and injects Risk Metrics.
    Standardizes on 1, 2, 3, 4, and 5-day forward return windows.
    """
    if perf_df.empty:
        print("⚠️ ARCHITECT NOTICE: No signals to analyze.")
        return

    # 1. DATA VALIDATION: Ensure 5-Day exists (Universal Standard)
    # If Day_5_$ is missing in your current fetch, this will alert you.
    windows = [1, 2, 3, 4, 5]
    available_days = [d for d in windows if f'Day_{d}_$' in perf_df.columns]

    # 2. STATISTICAL RIGOR CHECK
    total_signals = len(perf_df)
    print("\n" + "═" * 100)
    print(f"🚀 SENIOR ARCHITECT: ENHANCED PERFORMANCE & RISK AUDIT")
    print(f" Algorithm: {algorithm_name} | Trade Capital: ${trade_size:,.0f} | Lookback Days: {lookback_days}")
    if total_signals < 30:
        print(f"❌ STATISTICAL ALERT: Trade_Count ({total_signals}) < 30 threshold. This is likely due to the 5-day lockout reducing the number of valid signals. Results should be treated as anecdotal.")
    print("-" * 100)

    # Calculate capital exposure metrics once
    # Pass None for the unused price_df parameter to match original signature
    exposure_metrics = calculate_exposure_metrics(perf_df, _unused_price_df=None, trade_size=trade_size)

    # 3. VECTORIZED METRICS (Including Avg/Max at Risk - now capital exposure)
    summary_list = []

    for d in available_days:
        col = f'Day_{d}_$'

        # Retrieve the new capital exposure metrics
        avg_at_risk_capital = exposure_metrics.get(f'Avg_Risked_{d}D', 0)
        max_at_risk_capital = exposure_metrics.get(f'Max_Risked_{d}D', 0)

        summary_list.append({
            "Window": f"Day {d}",
            "Total Profit": f"${perf_df[col].sum():,.2f}",
            "Win Rate": f"{(perf_df[col] > 0).mean():.1%}",
            "Avg/Trade": f"${perf_df[col].mean():,.2f}",
            "Median": f"${perf_df[col].median():,.2f}",
            "Std Dev": f"${perf_df[col].std():,.2f}",
            "Avg At Risk": f"${avg_at_risk_capital:,.2f}",
            "Max At Risk": f"${max_at_risk_capital:,.2f}"
        })

    print(pd.DataFrame(summary_list).to_string(index=False))

    # 4. RESTORING TOP 5 ALPHA DRIVERS (Based on max available window)
    if available_days:
        best_window_col = f'Day_{max(available_days)}_$'
        print(f"\n🏆 TOP 5 ALPHA DRIVERS ({max(available_days)}-Day Cumulative):")
        top_5 = perf_df.sort_values(by=best_window_col, ascending=False).head(5)
        for _, row in top_5.iterrows():
            print(f" • {row['Symbol']}: ${row[best_window_col]:,.2f}")

    # 5. CRITIQUE ENGINE
    if 'Day_5_$' not in perf_df.columns:
        print("\n⚠️ COMPLIANCE WARNING: 5-Day Alpha window missing. Update fetch logic.")

    print("═" * 100)

In [5]:
# ==========================================
# ⚙️ RSI - OVERSOLD
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (16, 25),
    (24, 30),
    (14, 25),
    (22, 30),
    (18, 25),
    (22, 25),
    (26, 30)
]
CSV_FILE = "OptionVolume.csv"  # Source for your ticker universe
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from {CSV_FILE}")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ {CSV_FILE} not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI triggers using {len(RSI_CONFIG)} configurations since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            # Fetch data once per symbol
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # NEW: Loop through each Length/Threshold pair
            for length, threshold in RSI_CONFIG:
                # Calculate RSI for this specific length
                rsi_col = f'RSI_{length}'
                df[rsi_col] = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Check against the specific threshold for this length
                trigger_col = f'Trigger_{length}'
                df[trigger_col] = (df[rsi_col] < threshold) & (df[rsi_col].shift(1) >= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff]
                recent_hits = recent_hits[recent_hits[trigger_col] == True]

                for date, row in recent_hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": round(row['Close'], 2),
                        "RSI_Len": length,           # Added to track which pair triggered
                        "RSI_Val": round(row[rsi_col], 2),
                        "Thresh": threshold          # Added for clarity
                    })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*70)
    print(f"🚀 RSI MULTI-PAIR REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*70)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest) then Symbol
        results_df = results_df.sort_values(by=["Date", "Symbol"], ascending=[False, True])
        print(results_df.to_string(index=False))

    print("="*70)

✅ Loaded 100 symbols from OptionVolume.csv
🔍 Scanning for RSI triggers using 7 configurations since 2026-01-25...

🚀 RSI MULTI-PAIR REPORT: LAST 100 DAYS
Pairs Scanned: [(16, 25), (24, 30), (14, 25), (22, 30), (18, 25), (22, 25), (26, 30)]
      Date Symbol  Price  RSI_Len  RSI_Val  Thresh
2026-04-10    NKE  42.62       16    23.65      25
2026-04-10    NKE  42.62       18    24.52      25
2026-04-10    NOW  83.00       16    24.37      25
2026-04-10    NOW  83.00       24    29.08      30
2026-04-10    NOW  83.00       14    22.40      25
2026-04-10    NOW  83.00       22    28.23      30
2026-04-10    NOW  83.00       26    29.78      30
2026-04-10   SNOW 121.11       16    23.28      25
2026-04-10   SNOW 121.11       24    28.79      30
2026-04-10   SNOW 121.11       14    21.17      25
2026-04-10   SNOW 121.11       22    27.73      30
2026-04-10   SNOW 121.11       26    29.72      30
2026-04-10     ZS 118.05       22    29.84      30
2026-04-07    NKE  42.69       22    23.75    

In [6]:
# --- RSI Oversold Summary ---
if 'results_df' in locals() and not results_df.empty:
    profit_df = calculate_signal_profitability(results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 6)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${TRADE_CAPITAL:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
else:
    print("⚠️ No signals found in results_df to analyze.")

# --- Execution ---
display_strategy_summary(profit_df, algorithm_name='RSI Oversold', trade_size=TRADE_CAPITAL, lookback_days=LOOKBACK_DAYS)

 ʗ Analyzing profitability for 96 unique signals. Fetching historical data for 33 symbols...

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $5,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol  Price  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$
2026-01-29   INTU 501.43   -36.42  -142.30  -618.10  -514.47  -610.74
2026-01-29   NFLX  83.16    19.80   -24.00  -193.20  -180.00  -137.40
2026-01-29    NOW 116.73    11.76    53.34  -292.32  -237.72  -592.20
2026-01-30    IGV  90.31   -43.45  -270.60  -355.85  -585.20  -431.75
2026-02-02   HOOD  89.91  -156.20  -510.95  -947.65  -389.95  -184.25
2026-02-02   COIN 187.86  -213.20  -500.24 -1085.24  -591.24  -535.86
2026-02-03    CRM 195.89    76.31  -159.85  -125.43   -58.60   -73.07
2026-02-03   ADBE 271.93   140.04   -45.72   -63.90   -90.54  -130.68
2026-02-03   RDDT 165.41  -381.60  -430.80  -767.40  -710.10  -462.90
2026-02-03   SHOP 119.29  -216.07  -330.05  -296.84   -36.49   325.95
2026-02-03   SNOW 173.24  -222.60  -462.84  -134.6

In [7]:
# ==========================================
# ⚙️ RSI MOMENTUM
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (10, 80),
    (14, 75)
]
SMA_TREND = 200         # Baseline trend filter
CSV_FILE = "OptionVolume.csv"
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "COIN", "MSTR"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Wilder's Smoothing Precision (Yahoo Style)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is equivalent to Wilder's alpha=1/period
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_momentum_triggers():
    """Identifies 'Momentum Ignition' (RSI crossing ABOVE threshold)"""
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
    except:
        symbols = FALLBACK_SYMBOLS

    report_data = []

    # Fetch 350 days to ensure SMA 200 and RSI 10 are both stable
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 350)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🚀 Scanning for Momentum Ignition (RSI configurations: {RSI_CONFIG}) since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < SMA_TREND: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['SMA_200'] = df['Close'].rolling(window=SMA_TREND).mean()
            df['Trend'] = np.where(df['Close'] > df['SMA_200'], "UP", "DOWN")

            # NEW: Evaluate each specific RSI pair
            for length, threshold in RSI_CONFIG:
                rsi_vals = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Today is above threshold, Yesterday was below (Cross Above)
                trigger = (rsi_vals > threshold) & (rsi_vals.shift(1) <= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff].copy()
                recent_hits['Is_Trigger'] = trigger
                hits = recent_hits[recent_hits['Is_Trigger'] == True]

                for date, row in hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": f"${row['Close']:.2f}",
                        "RSI_Len": length,           # Track which length triggered
                        "RSI_Val": round(rsi_vals.loc[date], 1),
                        "Thresh": threshold,         # Track the target threshold
                        "Trend": row['Trend']
                    })
        except Exception:
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_momentum_triggers()

    print("\n" + "="*80)
    print(f"🚀 RSI MOMENTUM POWER ZONE REPORT (LAST {LOOKBACK_DAYS} DAYS)")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*80)

    if not triggers:
        print(f"No momentum breakouts detected in the last {LOOKBACK_DAYS} days.")
    else:
        max_results_df = pd.DataFrame(triggers)
        # Sort by Date (newest first)
        max_results_df = max_results_df.sort_values(by="Date", ascending=False)

        print(max_results_df.to_string(index=False))

    print("="*80)


🚀 Scanning for Momentum Ignition (RSI configurations: [(10, 80), (14, 75)]) since 2026-01-25...

🚀 RSI MOMENTUM POWER ZONE REPORT (LAST 100 DAYS)
Pairs Scanned: [(10, 80), (14, 75)]
      Date Symbol    Price  RSI_Len  RSI_Val  Thresh Trend
2026-05-05     BE  $295.66       10     80.8      80    UP
2026-05-05   TQQQ   $67.43       14     76.4      75    UP
2026-05-05    QQQ  $681.60       10     80.1      80    UP
2026-05-05    QQQ  $681.60       14     76.5      75    UP
2026-05-05   SNDK $1413.80       10     84.6      80    UP
2026-05-05   MRVL  $172.64       14     78.6      75    UP
2026-05-05   CSCO   $94.40       14     75.2      75    UP
2026-05-05    SMH  $525.76       10     81.3      80    UP
2026-05-05   SOXL  $146.78       14     78.1      75    UP
2026-05-05   QCOM  $182.54       14     77.1      75    UP
2026-05-05    SMH  $525.76       14     78.9      75    UP
2026-05-04   SNDK $1255.86       14     76.5      75    UP
2026-05-04    WDC  $442.36       10     81.3      8

In [8]:
# --- RSI Momentum Summary ---
if 'max_results_df' in locals() and not max_results_df.empty:
    profit_df = calculate_signal_profitability(max_results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 6)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${TRADE_CAPITAL:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
else:
    print("⚠️ No signals found in results_df to analyze.")

# --- Execution ---
display_strategy_summary(profit_df, algorithm_name='RSI Momentum', trade_size=TRADE_CAPITAL, lookback_days=LOOKBACK_DAYS)

 ʗ Analyzing profitability for 136 unique signals. Fetching historical data for 48 symbols...

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $5,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol    Price Trend  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$
2026-01-26    SLV   $98.34    UP   162.50   363.00   361.50 -1145.00 -1295.00
2026-01-26   ASML $1408.41    UP   123.29    28.62   125.00    28.85    83.83
2026-01-27     MU  $410.07    UP   300.33   306.44    55.63   330.55   110.33
2026-01-27    XOM  $135.92    UP    26.97   131.75   163.57    56.29   246.90
2026-01-27    GLW  $109.54    UP  -245.44  -302.93  -291.71    27.65   136.79
2026-01-28    FCX   $63.49    UP   116.76  -264.60  -223.35    80.96  -137.74
2026-01-28    WDC  $279.57    UP   -21.98  -500.82  -160.98   179.03  -174.91
2026-01-28   SNDK  $527.63    UP   105.03   437.58  1238.49  1510.92   512.28
2026-01-28    STX  $442.16    UP    39.96  -386.98  -109.60    16.68  -266.85
2026-01-29   LRCX  $247.87    UP  -293.